# Voting, Ensemble and Stacking Learnings


# Voting Classifier

In [47]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap 
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge 
from sklearn.ensemble import (
    VotingClassifier, 
    VotingRegressor,
    StackingClassifier,
    StackingRegressor,
    RandomForestClassifier

) 
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, 
    r2_score, 
    mean_squared_error,
    classification_report
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, MinMaxScaler
from sklearn.pipeline import Pipeline
from tabulate import tabulate
from xgboost import XGBClassifier



# Dataset Loading

In [48]:
df = pd.read_csv("healthcare_dataset.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 55500 entries, 0 to 55499
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Name                55500 non-null  str    
 1   Age                 55500 non-null  int64  
 2   Gender              55500 non-null  str    
 3   Blood Type          55500 non-null  str    
 4   Medical Condition   55500 non-null  str    
 5   Date of Admission   55500 non-null  str    
 6   Doctor              55500 non-null  str    
 7   Hospital            55500 non-null  str    
 8   Insurance Provider  55500 non-null  str    
 9   Billing Amount      55500 non-null  float64
 10  Room Number         55500 non-null  int64  
 11  Admission Type      55500 non-null  str    
 12  Discharge Date      55500 non-null  str    
 13  Medication          55500 non-null  str    
 14  Test Results        55500 non-null  str    
dtypes: float64(1), int64(2), str(12)
memory usage: 6.4 MB


In [49]:
df = df.sample(n=10000, random_state=42) # take only 10,000 elements out of the dataset

In [50]:
df["Test Results"].value_counts() # the target column

Test Results
Abnormal        3373
Normal          3356
Inconclusive    3271
Name: count, dtype: int64

In [51]:
df["Test Results"] = df["Test Results"].map({
    "Normal":0,
    "Abnormal":1,
    "Inconclusive":2
})

In [52]:
TARGET_COL = "Test Results"

X = df.drop(columns = ["Name", "Doctor", "Hospital", "Room Number", TARGET_COL])
y = df[TARGET_COL]

# Train/Test split

In [53]:
X_train, X_test, y_train,y_test = train_test_split(
    X, y, test_size = 0.2, random_state=42
)

In [54]:
categorical_features = X.select_dtypes(exclude = [np.number]).columns.tolist()
numerical_features = [col for col in X.columns if col not in categorical_features]

# Pipeline Stage

In [55]:
numerical_transformer = Pipeline(steps = [
    ("scaler", MinMaxScaler())
])
categorical_transformer = Pipeline(steps = [
    ("encoder", OrdinalEncoder(handle_unknown = "use_encoded_value", unknown_value = -1))
])

# ColumnTransformer Stage

In [56]:
preprocessor = ColumnTransformer(
    transformers =[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# Voting Classifier

In [57]:
lr = LogisticRegression(max_iter = 1000)
rf = RandomForestClassifier()
svc = SVC(probability = True)
xgb = XGBClassifier(
    n_estimators = 200,
    learning_rate = 0.05, 
    max_depth = 6,
    subsample = 0.8,
    colsample_bytree = 0.8,
    n_jobs = -1,
    random_state = 42
)


#  Voting Classifier --> Hard

In [58]:
hard_voting = VotingClassifier(
    estimators = [("lr", lr), ("rf", rf), ("svc", svc), ("xgb", xgb)],
    voting = "hard",
    n_jobs=-1
)



In [59]:
full_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", hard_voting)
])

In [60]:
full_pipeline.fit(X_train, y_train)
y_pred_hard = full_pipeline.predict(X_test)
print("Hard Voting Accuracy: ", classification_report(y_test, y_pred_hard))

/Users/murodjongafforov/Desktop/ensemble_metadata/ensemble_metadata/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/murodjongafforov/Desktop/ensemble_metadata/ensemble_metadata/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Hard Voting Accuracy:                precision    recall  f1-score   support

           0       0.37      0.52      0.43       703
           1       0.35      0.35      0.35       686
           2       0.34      0.19      0.24       611

    accuracy                           0.36      2000
   macro avg       0.35      0.35      0.34      2000
weighted avg       0.35      0.36      0.35      2000



# Voting Classifier --> SOFT

In [61]:
soft_voting = VotingClassifier(
    estimators = [("lr", lr), ("rf", rf), ("svc", svc), ("xgb", xgb)],
    voting = "soft",
    n_jobs=-1
)

In [62]:
full_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", soft_voting)
])

In [63]:
full_pipeline.fit(X_train, y_train)
y_pred_hard = full_pipeline.predict(X_test)
print("Hard Voting Accuracy: ", classification_report(y_test, y_pred_hard))

/Users/murodjongafforov/Desktop/ensemble_metadata/ensemble_metadata/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/murodjongafforov/Desktop/ensemble_metadata/ensemble_metadata/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Hard Voting Accuracy:                precision    recall  f1-score   support

           0       0.36      0.35      0.35       703
           1       0.37      0.34      0.36       686
           2       0.35      0.40      0.37       611

    accuracy                           0.36      2000
   macro avg       0.36      0.36      0.36      2000
weighted avg       0.36      0.36      0.36      2000



# Ensemble Classifier

In [64]:
rf = RandomForestClassifier(random_state=42)
full_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("rf_classifier", rf)
])

In [68]:
full_pipeline.fit(X_train, y_train)
y_pred_rf = full_pipeline.predict(X_test)
print("Random Forest Classifier: ", classification_report(y_test, y_pred_rf))

Random Forest Classifier:                precision    recall  f1-score   support

           0       0.37      0.36      0.37       703
           1       0.37      0.37      0.37       686
           2       0.34      0.35      0.35       611

    accuracy                           0.36      2000
   macro avg       0.36      0.36      0.36      2000
weighted avg       0.36      0.36      0.36      2000



# Stacking Classifier

In [72]:
# Level - 0 (Base Model)
base_learners = [
    ("lr", LogisticRegression()),
    ("rf", RandomForestClassifier()),
    ("svc", SVC()),
    ("xgb", XGBClassifier(
        n_estimators = 100,
        learning_rate = 0.01,
        max_depth = 5,
        subsample = 0.8,
        colsample_bytree = 0.8,
        n_jobs = -1,
        random_state = 42     
    ))
]


In [73]:
# Level - 1 (Meta Model)
meta_model = LogisticRegression() #Here only we want to measure by LogisticRegression
stack = StackingClassifier(estimators = base_learners, final_estimator = meta_model, n_jobs=-1)

stacking_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("stacking", stack)
])

stacking_pipeline.fit(X_train, y_train)
y_pred_stack = stacking_pipeline.predict(X_test)
print("Stacking Classifier MSE: ", mean_squared_error(y_test, y_pred_stack))
print("Stacking Classifier R2 score: ", r2_score(y_test, y_pred_stack))


/Users/murodjongafforov/Desktop/ensemble_metadata/ensemble_metadata/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/murodjongafforov/Desktop/ensemble_metadata/ensemble_metadata/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want 

Stacking Classifier MSE:  1.2835
Stacking Classifier R2 score:  -0.959889079592722


# Tabulate for transperent reporting 

In [78]:
from tabulate import tabulate

# Jadval ma'lumotlari
table_data = [
    [
        "Complexity (Time)", 
        "6.6s", 
        "3.7s", 
        "0.7s"
    ],
    [
        "Learning Combination", 
        "No", 
        "Yes", 
        "No"
    ],
    [
        "Risk of Overfitting", 
         "Low", 
        "Low", 
        "Low"
    ],
    [
        "Performance", 
        "Good", 
        "Worse", 
        "Good"
    ]
]

# Ustun nomlari
headers = ["Aspect", "Voting Classifier", "Stacking Classifier", "Random Forest"]

# Grid formatida konsolga chiqarish
print(tabulate(table_data, headers=headers, tablefmt="grid"))

+----------------------+---------------------+-----------------------+-----------------+
| Aspect               | Voting Classifier   | Stacking Classifier   | Random Forest   |
+======================+=====================+=======================+=================+
| Complexity (Time)    | 6.6s                | 3.7s                  | 0.7s            |
+----------------------+---------------------+-----------------------+-----------------+
| Learning Combination | No                  | Yes                   | No              |
+----------------------+---------------------+-----------------------+-----------------+
| Risk of Overfitting  | Low                 | Low                   | Low             |
+----------------------+---------------------+-----------------------+-----------------+
| Performance          | Good                | Worse                 | Good            |
+----------------------+---------------------+-----------------------+-----------------+
